# **api_client.py**

- Importa requests e pandas
- Define função de extração de precos dos bitcoins
- Gera DataFrame com datas e preços


In [1]:
%%writefile api_client.py

import requests
import pandas as pd

def get_bitcoin_prices(days=30):
    url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"
    params = {
        "vs_currency": "usd",
        "days": days
    }

    try:
      response = requests.get(url, params=params, timeout=10)
      response.raise_for_status()

      data = response.json()
      prices = data["prices"]

      df = pd.DataFrame(prices, columns=["timestamp", "price"])
      return df

    except requests.exceptions.RequestException as e:
        print(f"Erro na requisição da API: {e}")
        return None

Writing api_client.py


# **normalizacao.py**

- Define a função normalize para receber o DataFrame (df)
- Converte data e hora
- Remove os horários, mantendo apenas datas
- Agrupa dados por data e faz a média

In [ ]:
%%writefile normalizacao.py

import pandas as pd

def normalize(df):
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["date"] = df["date"].dt.date
    df = df.groupby("date").mean().reset_index()
    return df[["date", "price"]]

Writing normalizacao.py


# **analytics.py**
- Define a função que recebe o DataFrame normalizado
- Clacula a performance
- Organza a tabela por data decrescente
- Cria nova coluna 'retorn'

In [ ]:
%%writefile analytics.py

import pandas as pd

def calculate_returns(df):
    df = df.sort_values("date")
    df["return"] = df["price"].pct_change()
    return df

Writing analytics.py


# **reconciliador.py**
- Compara segunda fonte
- Cria cópia do primeiro df
- Ajuste de preços em 0,05% ( fator 1,0005)

In [ ]:
%%writefile reconciliador.py

def simulate_second_source(df):
    df2 = df.copy()
    df2["price"] = df2["price"] * 1.0005  # pequena variação simulada
    return df2

def reconcile(df1, df2):
    merged = df1.merge(df2, on="date", suffixes=("_source1", "_source2"))

    merged["price_diff"] = merged["price_source1"] - merged["price_source2"]
    merged["abs_diff"] = merged["price_diff"].abs()

    return merged

Writing reconciliador.py


# **database.py**
- Importa biblioteca de conexão com banco de dados
- Define função para salvar dados

In [ ]:
%%writefile database.py

from sqlalchemy import create_engine

def save_to_db(df, table_name="bitcoin_reconciliation"):
    engine = create_engine("sqlite:///market_data.db")
    df.to_sql(table_name, engine, if_exists="replace", index=False)

Writing database.py


# **main.py**
- Busca cada módulo e executa cada função determinada



In [ ]:
%%writefile main.py

from api_client import get_bitcoin_prices
from normalizacao import normalize
from analytics import calculate_returns
from reconciliador import simulate_second_source, reconcile
from database import save_to_db

def main():
  # 1. Buscar dados
  df = get_bitcoin_prices(30)

  # 2. Normalizar
  df = normalize(df)

  # 3. Calcular retorno
  df = calculate_returns(df)

  # 4. Simular segunda fonte
  df2 = simulate_second_source(df)

  # 5. Reconciliar
  result = reconcile(df, df2)

  # 6. Salvar CSV
  result.to_csv("resultado_final.csv", index=False)

  # 7. Salvar no banco
  save_to_db(result)

  print("Executado com sucesso")

if __name__ == "__main__":
  main()

Writing main.py


In [ ]:
!python main.py

Executado com sucesso
